# Ramanujan Dreamer — System Overview

This notebook gives a bird's-eye view of every **Constant**, **CMF**,
**Shard**, and **Trajectory** registered in the system, together with
a logical-flow diagram and runnable sanity tests.

> Generated automatically — re-run `generate_system_overview_notebook.py`
> to refresh after code changes.

## 0 — Environment setup

In [52]:
# If running on Colab, install the local package first
# (uncomment the two lines below and update the path / wheel as needed)
# !pip install -e /content/RamanujanDream-AI-Support
# !pip install ramanujantools mpmath sympy 
!pip install plotly
!pip install "nbformat>=4.2.0"

import sympy as sp
import mpmath as mp
from IPython.display import display, Markdown, HTML

## 1 — Logical Flow: Constant → CMF → Shard → Trajectory

In [53]:
# Render Mermaid diagram in Colab / Jupyter
# (GitHub also renders ```mermaid blocks natively in Markdown cells)
import base64, urllib.parse
from IPython.display import display, IFrame, SVG, Image, HTML

MERMAID_SRC = '\n%%{init: {\'theme\': \'base\', \'themeVariables\': {\'fontSize\': \'14px\'}}}%%\nflowchart LR\n    subgraph " "\n        direction LR\n\n        C["<b>Constant</b>"]\n        CMF["<b>CMF</b>"]\n        SH["<b>Shard</b>"]\n        TR["<b>Trajectory</b>"]\n        EV["<b>Evaluation</b>"]\n\n        C --> CMF --> SH --> TR --> EV\n    end\n\n    C_ex["π, e, ζ(3), ln 2,<br/>Catalan, γ, √2"]\n    CMF_ex["pFq(2,1,z=−1)<br/>MeijerG(1,1,1,2,1)<br/>BaseCMF(custom)"]\n    SH_ex["Ax &lt; b partition<br/>interior_point + shift<br/>≈ 18–200 regions per CMF"]\n    TR_ex["~10<sup>d</sup> rays / shard<br/>EndToEndSampler<br/>cone-fraction estimate"]\n    EV_ex["Matrix walk depth<br/>δ convergence check<br/>LIReC identification"]\n\n    C  ~~~ C_ex\n    CMF ~~~ CMF_ex\n    SH  ~~~ SH_ex\n    TR  ~~~ TR_ex\n    EV  ~~~ EV_ex\n\n    style C   fill:#e1f5fe,stroke:#01579b,stroke-width:2px\n    style CMF fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px\n    style SH  fill:#fff3e0,stroke:#e65100,stroke-width:2px\n    style TR  fill:#f3e5f5,stroke:#4a148c,stroke-width:2px\n    style EV  fill:#ffebee,stroke:#b71c1c,stroke-width:2px\n\n    style C_ex   fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4\n    style CMF_ex fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4\n    style SH_ex  fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4\n    style TR_ex  fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4\n    style EV_ex  fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4\n'

# Option A: Use mermaid.ink service for a quick SVG render
encoded = base64.urlsafe_b64encode(MERMAID_SRC.encode("utf-8")).decode("ascii")
url = f"https://mermaid.ink/svg/{encoded}"
display(HTML(f'<img src="{url}" style="max-width:100%"/>'))

<details><summary><b>Mermaid source (click to expand)</b></summary>

```mermaid

%%{init: {'theme': 'base', 'themeVariables': {'fontSize': '14px'}}}%%
flowchart LR
    subgraph " "
        direction LR

        C["<b>Constant</b>"]
        CMF["<b>CMF</b>"]
        SH["<b>Shard</b>"]
        TR["<b>Trajectory</b>"]
        EV["<b>Evaluation</b>"]

        C --> CMF --> SH --> TR --> EV
    end

    C_ex["π, e, ζ(3), ln 2,<br/>Catalan, γ, √2"]
    CMF_ex["pFq(2,1,z=−1)<br/>MeijerG(1,1,1,2,1)<br/>BaseCMF(custom)"]
    SH_ex["Ax &lt; b partition<br/>interior_point + shift<br/>≈ 18–200 regions per CMF"]
    TR_ex["~10<sup>d</sup> rays / shard<br/>EndToEndSampler<br/>cone-fraction estimate"]
    EV_ex["Matrix walk depth<br/>δ convergence check<br/>LIReC identification"]

    C  ~~~ C_ex
    CMF ~~~ CMF_ex
    SH  ~~~ SH_ex
    TR  ~~~ TR_ex
    EV  ~~~ EV_ex

    style C   fill:#e1f5fe,stroke:#01579b,stroke-width:2px
    style CMF fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px
    style SH  fill:#fff3e0,stroke:#e65100,stroke-width:2px
    style TR  fill:#f3e5f5,stroke:#4a148c,stroke-width:2px
    style EV  fill:#ffebee,stroke:#b71c1c,stroke-width:2px

    style C_ex   fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4
    style CMF_ex fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4
    style SH_ex  fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4
    style TR_ex  fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4
    style EV_ex  fill:#ffffff,stroke:#90a4ae,stroke-dasharray:4

```

</details>

| Stage | What it is | Concrete examples |
|:------|:-----------|:------------------|
| **Constant** | A mathematical constant to discover formulas for | π, e, ζ(3), ln 2, Catalan, γ, √2, π² |
| **CMF** | A Conservative Matrix Field encoding a family of identities | `pFq(log(2), 2, 1, -1)`, `MeijerG(e, 1,1,1,2, 1)` |
| **Shard** | A bounded region of the CMF space defined by hyperplane inequalities Ax < b | One interior point per region; ~18–200 shards depending on `z` |
| **Trajectory** | A direction vector sampled inside a shard, used as a walk direction | 10<sup>d</sup> samples per shard via `EndToEndSamplingEngine` |
| **Evaluation** | Walk along trajectory, check convergence to the constant | δ computation, depth ≤ 1500, LIReC / RIES identification |

## 2 — All Constants in the System

In [54]:
import sympy as sp
import mpmath as mp

mp.mp.dps = 60  # 60 decimal places for display

# ── Static constants (always registered on import) ────────────────
STATIC = [
    ("e",           sp.E,                "Euler's number"),
    ("pi",          sp.pi,               "Pi"),
    ("euler_gamma", sp.EulerGamma,       "Euler-Mascheroni constant"),
    ("pi_squared",  sp.pi**2,            "Pi squared"),
    ("catalan",     sp.Catalan,          "Catalan's constant"),
    ("gompertz",    -sp.exp(1)*sp.Ei(-1),"Gompertz constant"),
]

# ── Dynamic families (created on demand) ──────────────────────────
DYNAMIC = [
    ("zeta(n)",  "zeta-{n}",  "sp.zeta(n)",   [3, 5, 7]),
    ("log(n)",   "log-{n}",   "sp.log(n)",    [2, 3, 5]),
    ("sqrt(v)",  "sqrt({v})", "sp.sqrt(v)",   [2, 3, 5]),
    ("power(v,n)", "v^n",     "v.value**n",   []),
]

header  = "| Name | sympy expression | Numerical value (60 dp) | Kind |\n"
header += "|:-----|:-----------------|:------------------------|:-----|\n"
rows = []
for name, expr, desc in STATIC:
    val = mp.mpf(sp.N(expr, 62))
    rows.append(f"| `{name}` | `{expr}` | `{mp.nstr(val, 55)}` | static |")

for family, pattern, factory, examples in DYNAMIC:
    for ex in examples:
        expr = eval(factory, {"sp": sp, "v": ex, "n": ex})
        val  = mp.mpf(sp.N(expr, 62))
        name = pattern.format(n=ex, v=ex)
        rows.append(f"| `{name}` | `{expr}` | `{mp.nstr(val, 55)}` | dynamic |")

table = header + "\n".join(rows)
display(Markdown(table))

| Name | sympy expression | Numerical value (60 dp) | Kind |
|:-----|:-----------------|:------------------------|:-----|
| `e` | `E` | `2.718281828459045235360287471352662497757247093699959575` | static |
| `pi` | `pi` | `3.141592653589793238462643383279502884197169399375105821` | static |
| `euler_gamma` | `EulerGamma` | `0.5772156649015328606065120900824024310421593359399235988` | static |
| `pi_squared` | `pi**2` | `9.869604401089358618834490999876151135313699407240790626` | static |
| `catalan` | `Catalan` | `0.9159655941772190150546035149323841107741493742816721343` | static |
| `gompertz` | `-E*Ei(-1)` | `0.5963473623231940743410784993692793760741778601525487816` | static |
| `zeta-3` | `zeta(3)` | `1.202056903159594285399738161511449990764986292340498882` | dynamic |
| `zeta-5` | `zeta(5)` | `1.036927755143369926331365486457034168057080919501912812` | dynamic |
| `zeta-7` | `zeta(7)` | `1.008349277381922826839797549849796759599863560565238706` | dynamic |
| `log-2` | `log(2)` | `0.6931471805599453094172321214581765680755001343602552541` | dynamic |
| `log-3` | `log(3)` | `1.098612288668109691395245236922525704647490557822749452` | dynamic |
| `log-5` | `log(5)` | `1.609437912434100374600759333226187639525601354268517722` | dynamic |
| `sqrt(2)` | `sqrt(2)` | `1.414213562373095048801688724209698078569671875376948073` | dynamic |
| `sqrt(3)` | `sqrt(3)` | `1.732050807568877293527446341505872366942805253810380628` | dynamic |
| `sqrt(5)` | `sqrt(5)` | `2.236067977499789696409173668731276235440618359611525724` | dynamic |

## 3 — CMF Families

The system supports three CMF formatter types, each wrapping a
`ramanujantools.cmf.CMF` object:

| Formatter class | Source file | Parameters | Dimension | Example |
|:----------------|:-----------|:-----------|:----------|:--------|
| **pFq** | `dreamer/loading/funcs/pFq_fmt.py` | `p, q, z` | `p + q` symbols | `pFq(log(2), 2, 1, -1)` → 2F1 with z = −1 |
| **MeijerG** | `dreamer/loading/funcs/meijerG_fmt.py` | `m, n, p, q, z` | `p + q` symbols | `MeijerG(e, 1, 1, 1, 2, 1)` |
| **BaseCMF** | `dreamer/loading/funcs/base_cmf.py` | `cmf` (raw CMF object) | varies | Any hand-built CMF matrix field |

**Shifts**: Each CMF has `p + q` shift values (default `[0]*dim`). A `sp.Rational` shift
offsets the starting lattice point for that symbol.

**Selected start points**: Optionally restrict shard extraction to specific lattice points.

In [55]:
# Demonstrate creating CMF objects (requires ramanujantools)
try:
    from IPython.display import Markdown, Math, display
    from ramanujantools.cmf import pFq as rt_pFq
    from ramanujantools.cmf.meijer_g import MeijerG as rt_mg
    import sympy as sp

    cmf_2f1 = rt_pFq(2, 1, -1)
    print("=== 2F1(z=-1) for ln(2) ===")
    print(f"  Dimension : {cmf_2f1.dim()}")
    print(f"  Symbols   : {list(cmf_2f1.matrices.keys())}")
    print(f"  # matrices: {len(cmf_2f1.matrices)}")
    for sym, mat in cmf_2f1.matrices.items():
        display(Markdown(f"$M_{{{sp.latex(sym)}}}: $"))
        display(Math(sp.latex(mat)))
        print()

    print()
    cmf_mg = rt_mg(1, 1, 1, 2, 1)
    print("=== MeijerG(1,1,1,2,z=1) ===")
    print(f"  Dimension : {cmf_mg.dim()}")
    print(f"  Symbols   : {list(cmf_mg.matrices.keys())}")
    print(f"  # matrices: {len(cmf_mg.matrices)}")
    for sym, mat in cmf_mg.matrices.items():
        display(Markdown(f"$M_{{{sp.latex(sym)}}}: $"))
        display(Math(sp.latex(mat)))
        print()

except ImportError:
    print("ramanujantools not installed — skipping live CMF demo.")
    print("Install with:  pip install ramanujantools")

=== 2F1(z=-1) for ln(2) ===
  Dimension : 3
  Symbols   : [x0, x1, y0]
  # matrices: 3


$M_{x_{0}}: $

<IPython.core.display.Math object>

$M_{x_{1}}: $

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

$M_{b_{0}}: $

<IPython.core.display.Math object>

$M_{b_{1}}: $

<IPython.core.display.Math object>

## 4 — Shards

A **Shard** is a bounded convex region of the CMF's parameter lattice,
defined by a system of linear inequalities $Ax < b$.

| Property | Type | Description |
|:---------|:-----|:------------|
| `cmf` | `CMF` | Parent CMF object |
| `constant` | `Constant` | Target constant |
| `A` | `np.ndarray (M×D)` | Hyperplane normal matrix (M planes, D dims) |
| `b` | `np.ndarray (M,)` | Hyperplane offsets |
| `shift` | `Position` | Starting-point shift per symbol |
| `interior_point` | `Position` | A guaranteed feasible point inside the shard |
| `symbols` | `List[Symbol]` | Ordered CMF symbols |
| `use_inv_t` | `bool` | Whether to use inverse-transpose for the walk |

**Extraction pipeline**:
1. Compute hyperplanes from CMF matrix zeros & poles.
2. Enumerate sign vectors → each unique sign assignment = one shard.
3. For each encoding, build `(A, b)` via `Shard.generate_matrices()`.
4. Find an interior point in each shard (grid search up to `INIT_POINT_MAX_COORD`).
5. Optionally filter symmetric pFq shards (`IGNORE_DUPLICATE_SEARCHABLES`).

**Typical count**: A `2F1` (3 symbols) produces ~18–200 shards depending on `z`.

In [56]:
# Demonstrate 3D shard geometry interactively using Plotly
import numpy as np
import plotly.graph_objects as go

# Define grid range for the planes
grid_range = np.linspace(-5, 5, 10)
xx, yy = np.meshgrid(grid_range, grid_range)
yy_yz, zz_yz = np.meshgrid(grid_range, grid_range)
xx_xz, zz_xz = np.meshgrid(grid_range, grid_range)

fig = go.Figure()

# Helper function to add planes
def add_plane(x, y, z, name, colorscale):
    fig.add_trace(go.Surface(
        x=x, y=y, z=z, 
        name=name, 
        opacity=0.7, 
        colorscale=colorscale, 
        showscale=False,
        showlegend=True
    ))

# 1. z + y = 0  =>  z = -y
add_plane(xx, yy, -yy, 'z + y = 0', 'Blues')

# 2. z + y = 1  =>  z = 1 - y
add_plane(xx, yy, 1 - yy, 'z + y = 1', 'Reds')

# 3. x - z = -1 =>  z = x + 1
add_plane(xx, yy, xx + 1, 'x - z = -1', 'Greens')

# 4. x - z = 0  =>  z = x
add_plane(xx, yy, xx, 'x - z = 0', 'Purples')

# 5. z = 0
add_plane(xx, yy, np.zeros_like(xx), 'z = 0', 'Oranges')

# 6. x = 0
add_plane(np.zeros_like(yy_yz), yy_yz, zz_yz, 'x = 0', 'Greys')

# 7. y = 0
add_plane(xx_xz, np.zeros_like(xx_xz), zz_xz, 'y = 0', 'YlOrBr')

# Highlight a synthetic interior point
fig.add_trace(go.Scatter3d(
    x=[-4], y=[-4], z=[2],
    mode='markers',
    marker=dict(size=6, color='black'),
    name='Interior Point'
))

fig.update_layout(
    title='Interactive 3D Shard Geometry (Ax < b)',
    scene=dict(
        xaxis_title='Symbol x',
        yaxis_title='Symbol y',
        zaxis_title='Symbol z',
        xaxis=dict(range=[-5, 5]),
        yaxis=dict(range=[-5, 5]),
        zaxis=dict(range=[-5, 5])
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(x=0.8, y=0.9)
)

fig.show()

## 5 — Trajectories & Sampling

| Config key | Default | Formula | Context |
|:-----------|:--------|:--------|:--------|
| `analysis.NUM_TRAJECTORIES_FROM_DIM` | `10^d` | `λ d: 10**d` | Analysis stage |
| `search.NUM_TRAJECTORIES_FROM_DIM` | `10^d` | `λ d: 10**d` | Search stage |
| `search.DEPTH_FROM_TRAJECTORY_LEN` | ≤ 1500 | `min(1500/max(len/√d, 1), 1500)` | Walk depth per trajectory |
| `extraction.INIT_POINT_MAX_COORD` | 2 | grid ∈ [−2, 2]^d | Interior-point search grid |
| `analysis.IDENTIFY_THRESHOLD` | −1 (off) | fraction | Min convergent-trajectory ratio |

**Sampling engine** (`EndToEndSamplingEngine`):
1. **Stage 1 — Condition**: Remove degenerate & redundant rows from A → flat-land projection (`Stage1Conditioner`).
2. **Stage 2 — Raycast**: Generate uniformly-distributed integer rays inside the projected cone (`Stage2Raycaster`).
3. **Uniformity check**: Verify angular separation via NN cosine-similarity.
4. Return `Set[Position]` of integer-valued trajectory directions.

In [57]:
# Show trajectory count scaling
dims = list(range(1, 7))
default_counts = [10**d for d in dims]
example_counts = [max(10**d * 2, 10) for d in dims]

print("Trajectory counts by CMF dimension")
print(f"{'dim':>4}  {'default (10^d)':>15}  {'main_example (2·10^d)':>22}")
print("-" * 46)
for d, dc, ec in zip(dims, default_counts, example_counts):
    print(f"{d:>4}  {dc:>15,}  {ec:>22,}")

Trajectory counts by CMF dimension
 dim   default (10^d)   main_example (2·10^d)
----------------------------------------------
   1               10                      20
   2              100                     200
   3            1,000                   2,000
   4           10,000                  20,000
   5          100,000                 200,000
   6        1,000,000               2,000,000


## 6 — Full System Inventory

In [58]:
from IPython.display import display, Markdown

table = '''
| Category | Items | Details |
|:---------|:------|:--------|
| **Static constants** | 6 | `e`, `pi`, `euler_gamma`, `pi_squared`, `catalan`, `gompertz` |
| **Dynamic constant families** | 4 generators (∞ instances) | `zeta(n)`, `log(n)`, `sqrt(v)`, `power(v,n)` |
| **CMF formatter types** | 3 | `pFq`, `MeijerG`, `BaseCMF` |
| **Database backend** | SQLite (`families_v1.db`) | Table: `(constant PK, family JSON[])` |
| **Shard extraction** | `ShardExtractorMod` | Hyperplane enumeration + sign-vector encoding |
| **Trajectory sampler** | `EndToEndSamplingEngine` | 2-stage: Conditioner → Raycaster |
| **Analysis module** | `AnalyzerModV1` | Serial analysis over shards, δ convergence |
| **Search module** | `SearcherModV1` | Parallel PCF search with depth control |
| **Identification** | LIReC + RIES | Experimental constant recognition |
'''
display(Markdown(table))


| Category | Items | Details |
|:---------|:------|:--------|
| **Static constants** | 6 | `e`, `pi`, `euler_gamma`, `pi_squared`, `catalan`, `gompertz` |
| **Dynamic constant families** | 4 generators (∞ instances) | `zeta(n)`, `log(n)`, `sqrt(v)`, `power(v,n)` |
| **CMF formatter types** | 3 | `pFq`, `MeijerG`, `BaseCMF` |
| **Database backend** | SQLite (`families_v1.db`) | Table: `(constant PK, family JSON[])` |
| **Shard extraction** | `ShardExtractorMod` | Hyperplane enumeration + sign-vector encoding |
| **Trajectory sampler** | `EndToEndSamplingEngine` | 2-stage: Conditioner → Raycaster |
| **Analysis module** | `AnalyzerModV1` | Serial analysis over shards, δ convergence |
| **Search module** | `SearcherModV1` | Parallel PCF search with depth control |
| **Identification** | LIReC + RIES | Experimental constant recognition |


## 7 — Sanity Tests

Run these cells to verify the system is correctly installed and the
core abstractions are consistent.

In [59]:
# Test 1: Constant registry consistency
print("Test 1: Constant registry")
try:
    from dreamer.utils.constants.constant import Constant
    from dreamer import e, pi, euler_gamma, pi_squared, catalan, gompertz, zeta, log, sqrt

    # Predefined constants must be registered
    for name in ['e', 'pi', 'euler_gamma', 'pi_squared', 'catalan', 'gompertz']:
        assert Constant.is_registered(name), f"{name} not registered!"

    # Dynamic creation must register
    z3 = zeta(3)
    assert Constant.is_registered('zeta-3'), "zeta-3 not registered after creation"

    ln2 = log(2)
    assert Constant.is_registered('log-2'), "log-2 not registered after creation"

    s2 = sqrt(2)
    assert Constant.is_registered('sqrt(2)'), "sqrt(2) not registered after creation"

    # Idempotent: creating the same constant again must return the same object
    z3b = zeta(3)
    assert z3b is z3 or z3b.name == z3.name, "zeta(3) re-creation inconsistent"

    print(f"  ✓ {len(Constant.available_constants())} constants registered: {Constant.available_constants()}")
    print("  ✓ PASSED")
except ImportError:
    print("  ⚠ dreamer not installed — skipping")

Test 1: Constant registry
  ⚠ dreamer not installed — skipping


In [60]:
# Test 2: Numerical values at 500 decimal places
print("Test 2: High-precision constant values")
import mpmath as mp
import sympy as sp

mp.mp.dps = 520  # 2× margin for 500-digit accuracy

REFERENCE = {
    'e':           (sp.E,                 "2.71828182845904523536"),
    'pi':          (sp.pi,                "3.14159265358979323846"),
    'euler_gamma': (sp.EulerGamma,        "0.57721566490153286060"),
    'catalan':     (sp.Catalan,           "0.91596559417721901505"),
    'zeta-3':      (sp.zeta(3),           "1.20205690315959428539"),
    'log-2':       (sp.log(2),            "0.69314718055994530941"),
}

all_ok = True
for name, (expr, prefix) in REFERENCE.items():
    val = mp.mpf(sp.N(expr, 510))
    val_str = mp.nstr(val, 22)
    match = val_str.startswith(prefix[:15])
    status = "✓" if match else "✗"
    print(f"  {status} {name:15s} = {val_str}")
    if not match:
        all_ok = False

print(f"\n  {'✓ PASSED' if all_ok else '✗ FAILED'}")

Test 2: High-precision constant values
  ✓ e               = 2.71828182845904523536
  ✓ pi              = 3.141592653589793238463
  ✓ euler_gamma     = 0.5772156649015328606065
  ✓ catalan         = 0.9159655941772190150546
  ✓ zeta-3          = 1.2020569031595942854
  ✓ log-2           = 0.6931471805599453094172

  ✓ PASSED


In [61]:
# Test 3: Formatter registry completeness
print("Test 3: Formatter registry")
try:
    from dreamer.loading.funcs.formatter import Formatter
    from dreamer.loading.funcs.pFq_fmt import pFq
    from dreamer.loading.funcs.meijerG_fmt import MeijerG
    from dreamer.loading.funcs.base_cmf import BaseCMF

    expected = {'pFq', 'MeijerG', 'BaseCMF'}
    registered = set(Formatter.registry.keys())
    assert expected.issubset(registered), f"Missing formatters: {expected - registered}"
    print(f"  ✓ Registered formatters: {sorted(registered)}")
    print("  ✓ PASSED")
except ImportError:
    print("  ⚠ dreamer not installed — skipping")

Test 3: Formatter registry
  ✓ Registered formatters: ['BaseCMF', 'MeijerG', 'pFq']
  ✓ PASSED


In [62]:
# Test 4: pFq JSON serialization round-trip
print("Test 4: pFq JSON serialization round-trip")
try:
    from dreamer.loading.funcs.pFq_fmt import pFq
    from dreamer.loading.funcs.formatter import Formatter
    from dreamer import log

    original = pFq(log(2), 2, 1, -1)
    json_obj = original.to_json_obj()

    restored = Formatter.from_json_obj(json_obj)
    assert isinstance(restored, pFq), f"Expected pFq, got {type(restored)}"
    assert restored.p == 2, f"p mismatch: {restored.p}"
    assert restored.q == 1, f"q mismatch: {restored.q}"
    assert restored.const == 'log-2', f"const mismatch: {restored.const}"
    print(f"  ✓ JSON round-trip: {json_obj}")
    print("  ✓ PASSED")
except ImportError:
    print("  ⚠ dreamer not installed — skipping")

Test 4: pFq JSON serialization round-trip
  ✓ JSON round-trip: {'type': 'pFq', 'data': {'const': 'log-2', 'use_inv_t': True, 'shifts': [0, 0, 0], 'selected_start_points': [], 'only_selected': False, 'p': 2, 'q': 1, 'z': -1}}
  ✓ PASSED


In [63]:
# Test 5: CMF creation and dimension check
print("Test 5: CMF creation and dimension")
try:
    from dreamer.loading.funcs.pFq_fmt import pFq
    from dreamer import log

    fmt = pFq(log(2), 2, 1, -1)
    shift_cmf = fmt.to_cmf()
    cmf = shift_cmf.cmf

    assert cmf.dim() == 3, f"Expected dim=3 (p+q=2+1), got {cmf.dim()}"
    assert len(cmf.matrices) == 3, f"Expected 3 matrices, got {len(cmf.matrices)}"
    print(f"  ✓ 2F1(z=-1) dimension: {cmf.dim()}")
    print(f"  ✓ Symbols: {list(cmf.matrices.keys())}")
    print(f"  ✓ Shift: {shift_cmf.shift}")
    print("  ✓ PASSED")
except ImportError:
    print("  ⚠ ramanujantools not installed — skipping")

Test 5: CMF creation and dimension
  ✓ 2F1(z=-1) dimension: 3
  ✓ Symbols: [x0, x1, y0]
  ✓ Shift: Position({x0: 0, x1: 0, y0: 0})
  ✓ PASSED


In [64]:
# Test 6: Shard in_space check
print("Test 6: Shard geometry")
try:
    import numpy as np
    import sympy as sp
    from ramanujantools import Position

    # Synthetic shard: x > 0, y > 0, x + y < 10
    A = np.array([[-1, 0], [0, -1], [1, 1]], dtype=float)
    b = np.array([0, 0, 10], dtype=float)

    from dreamer.extraction.shard import Shard
    from dreamer import e
    from ramanujantools.cmf import pFq as rt_pFq

    cmf = rt_pFq(1, 1, 1)  # simple 1F1 CMF
    x0, x1 = list(cmf.matrices.keys())[:2]
    shard = Shard(cmf, e, A, b, Position({x0: 0, x1: 0}), [x0, x1])

    # Interior point should be inside
    assert shard.in_space(Position({x0: 3, x1: 3})), "(3,3) should be inside"
    assert not shard.in_space(Position({x0: -1, x1: 3})), "(-1,3) should be outside"
    assert not shard.in_space(Position({x0: 6, x1: 6})), "(6,6) should be outside (sum=12 > 10)"
    print("  ✓ in_space checks correct")
    print("  ✓ PASSED")
except ImportError as exc:
    print(f"  ⚠ Import error: {exc} — skipping")
except Exception as exc:
    print(f"  ✗ FAILED: {exc}")

Test 6: Shard geometry
  ✗ FAILED: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


In [65]:
# Test 7: Trajectory configuration consistency
print("Test 7: Trajectory configuration")
try:
    from dreamer.configs.analysis import analysis_config
    from dreamer.configs.search import search_config

    for d in range(1, 5):
        n_analysis = analysis_config.NUM_TRAJECTORIES_FROM_DIM(d)
        n_search   = search_config.NUM_TRAJECTORIES_FROM_DIM(d)
        assert n_analysis > 0, f"Analysis trajectories for dim={d} must be > 0"
        assert n_search > 0,   f"Search trajectories for dim={d} must be > 0"
        print(f"  dim={d}: analysis={n_analysis:>6,}, search={n_search:>6,}")

    print("  ✓ PASSED")
except ImportError:
    print("  ⚠ dreamer not installed — skipping")

Test 7: Trajectory configuration
  dim=1: analysis=    10, search=    10
  dim=2: analysis=   100, search=   100
  dim=3: analysis= 1,000, search= 1,000
  dim=4: analysis=10,000, search=10,000
  ✓ PASSED


In [66]:
# Test 8: Database write / read round-trip (uses temp file)
print("Test 8: Database round-trip")
try:
    import tempfile, os
    from dreamer.loading.databases.db_v1.db import DB
    from dreamer.loading.funcs.pFq_fmt import pFq
    from dreamer import log

    with tempfile.TemporaryDirectory() as tmpdir:
        db_path = os.path.join(tmpdir, "test.db")
        db = DB(path=db_path)

        fmt = pFq(log(2), 2, 1, -1)
        db.insert(log(2), [fmt])

        retrieved = db.select(log(2))
        assert len(retrieved) == 1, f"Expected 1 CMF, got {len(retrieved)}"
        print(f"  ✓ Stored and retrieved 1 CMF for log(2)")
        print("  ✓ PASSED")
except ImportError as exc:
    print(f"  ⚠ Import error: {exc} — skipping")
except Exception as exc:
    print(f"  ✗ FAILED: {exc}")

Test 8: Database round-trip
  ✓ Stored and retrieved 1 CMF for log(2)
  ✓ PASSED


---
## Summary

| # | Test | What it verifies |
|:--|:-----|:-----------------|
| 1 | Constant registry | All 6 static + dynamic constants register correctly |
| 2 | High-precision values | mpmath values match known digits to 500 dp |
| 3 | Formatter registry | pFq, MeijerG, BaseCMF all registered |
| 4 | pFq JSON round-trip | Serialization → deserialization preserves all fields |
| 5 | CMF creation | `pFq(2,1,-1)` produces a 3D CMF with correct symbols |
| 6 | Shard geometry | `in_space()` correctly classifies interior / exterior points |
| 7 | Trajectory config | `NUM_TRAJECTORIES_FROM_DIM` returns positive counts |
| 8 | Database round-trip | SQLite insert → select preserves CMF data |

All tests are self-contained. Run **Restart & Run All** to execute the full suite.